In [5]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


GPU free: 15.5 GB


In [6]:
# verify if drive is mounted
import os
print(os.listdir("/content/drive/MyDrive"))

['EarningScribe']


In [7]:
!pip install transformers peft accelerate bitsandbytes datasets \
             sentencepiece -U -q
print("Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.4 MB/s eta 0:00:00
Done


In [8]:
import os
!git clone https://github.com/jiviteshhp/EarningScribe.git /content/EarningScribe
%cd /content/EarningScribe
print("Repo cloned")
print(os.listdir("/content/EarningScribe/src"))

Cloning into '/content/EarningScribe'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 0), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 5.89 KiB | 2.95 MiB/s, done.
/content/EarningScribe
Repo cloned
['download_data.py', 'generate.py', 'rag.py']


In [9]:
!pip install transformers peft accelerate bitsandbytes datasets \
             sentencepiece evaluate rouge-score bert-score -q
print("Dependencies installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.2 MB/s eta 0:00:00
Dependencies installed


In [10]:
!pip install -U bitsandbytes>=0.46.1

In [11]:
!ls /content/EarningScribe/data/processed/

ls: cannot access '/content/EarningScribe/data/processed/': No such file or directory


In [12]:
import subprocess
subprocess.run(["pip", "install", "-U", "bitsandbytes", "-q"], check=True)

import os, json, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset

DRIVE_BASE = "/content/drive/MyDrive/EarningScribe"
DATA_BASE  = "/content/EarningScribe/data/processed"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "models/checkpoints")
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_BASE, exist_ok=True)

# ── data download if missing ──────────────────────────────────────────
if not os.path.exists(os.path.join(DATA_BASE, "train.json")):
    print("Downloading data...")

    def clean_text(text):
        skip = ["operator","safe harbor","forward-looking","this concludes",
                "thank you for joining","ladies and gentlemen",
                "this call is being recorded"]
        lines = [l.strip() for l in text.split("\n")
                 if len(l.strip()) >= 15
                 and not any(l.strip().lower().startswith(p) for p in skip)]
        return " ".join(lines)

    dataset  = load_dataset("lamini/earnings-calls-qa")
    samples  = []
    seen     = set()

    for i, item in enumerate(dataset["train"]):
        ans  = item["answer"].strip()
        txt  = clean_text(item["transcript"])
        if "i do not know" in ans.lower(): continue
        if len(txt.split()) < 50:          continue
        if len(ans.split()) < 5:           continue
        fp = txt[:200]
        if fp in seen: continue
        seen.add(fp)
        samples.append({
            "id": f"sample_{i}", "transcript": txt,
            "ticker": item.get("ticker",""), "date": item.get("date",""),
            "output": {"financial_insight": item["question"].strip(), "detail": ans}
        })
        if len(samples) == 5000: break

    n = len(samples)
    for name, sl in [("train", samples[:int(n*.8)]),
                     ("validation", samples[int(n*.8):int(n*.9)]),
                     ("test", samples[int(n*.9):])]:
        with open(os.path.join(DATA_BASE, f"{name}.json"), "w") as f:
            json.dump(sl, f)
        print(f"Saved {len(sl)} -> {name}.json")
else:
    print("Data already exists")

# ── dataset class ─────────────────────────────────────────────────────
class EarningsDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        with open(path) as f:
            self.data = json.load(f)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item   = self.data[idx]
        prompt = f"""You are a senior financial analyst.

TRANSCRIPT:
{item['transcript'][:600]}

Generate a structured JSON report with these fields:
- company_summary
- key_metrics (list of specific numbers)
- guidance
- risks
- sentiment (positive/neutral/negative)

REPORT:"""
        full  = prompt + json.dumps(item["output"])
        enc   = self.tokenizer(full, truncation=True,
                               max_length=self.max_length,
                               padding="max_length", return_tensors="pt")
        ids   = enc["input_ids"].squeeze()
        mask  = enc["attention_mask"].squeeze()
        labs  = ids.clone()
        labs[:len(self.tokenizer(prompt)["input_ids"])] = -100
        return {"input_ids": ids, "attention_mask": mask, "labels": labs}

# ── model + lora ──────────────────────────────────────────────────────
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map="auto", dtype=torch.float16
)
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    task_type=TaskType.CAUSAL_LM
))
model.print_trainable_parameters()

# ── training ──────────────────────────────────────────────────────────
train_ds = EarningsDataset(os.path.join(DATA_BASE,"train.json"), tokenizer)
val_ds   = EarningsDataset(os.path.join(DATA_BASE,"validation.json"), tokenizer)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=3,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, warmup_steps=50,
    learning_rate=1e-4, fp16=True, eval_strategy="steps",
    eval_steps=100, save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, logging_steps=25, report_to="none"
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
)

print("Starting training...")
trainer.train()

adapter_path = os.path.join(DRIVE_BASE, "models/lora_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved to {adapter_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

filtered_predictions.jsonl:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/860164 [00:00<?, ? examples/s]

Saved 4000 -> train.json
Saved 500 -> validation.json
Saved 500 -> test.json
Loading model...


...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
Train: 4000  Val: 500
Starting training...


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss,Validation Loss
100,0.267276,0.247374
200,0.238203,0.227791
300,0.225602,0.220863
400,0.219851,0.216944
500,0.207611,0.214459
600,0.220640,0.212951
700,0.230883,0.212228
750,0.220680,0.212095


Saved to /content/drive/MyDrive/EarningScribe/models/lora_adapter


In [15]:
import os

DRIVE_BASE   = "/content/drive/MyDrive/EarningScribe"
ADAPTER_PATH = os.path.join(DRIVE_BASE, "models/lora_adapter")
os.makedirs(ADAPTER_PATH, exist_ok=True)

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Files saved:")
for f in os.listdir(ADAPTER_PATH):
    print(f"  {f}")

Files saved:
  adapter_model.safetensors
  chat_template.jinja
  tokenizer.json
  adapter_config.json
  README.md
  tokenizer_config.json


In [16]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import evaluate

DRIVE_BASE   = "/content/drive/MyDrive/EarningScribe"
DATA_BASE    = "/content/EarningScribe/data/processed"
ADAPTER_PATH = os.path.join(DRIVE_BASE, "models/lora_adapter")
MODEL_NAME   = "Qwen/Qwen2-1.5B-Instruct"

# load test data
with open(os.path.join(DATA_BASE, "test.json")) as f:
    test_data = json.load(f)

# load base model + attach lora adapter
print("Loading fine-tuned model...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map="auto", dtype=torch.float16
)
# PeftModel loads your LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Model loaded")


def generate_report(transcript):
    prompt = f"""You are a senior financial analyst.

TRANSCRIPT:
{transcript[:600]}

Generate a structured JSON report with these fields:
- company_summary
- key_metrics (list of specific numbers)
- guidance
- risks
- sentiment (positive/neutral/negative)

REPORT:"""

    inputs  = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=768).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# run on 50 test samples
print("Running evaluation on 50 test samples...")
predictions = []
references  = []

for i, item in enumerate(test_data[:50]):
    raw       = generate_report(item["transcript"])
    reference = item["output"]["detail"]
    predictions.append(raw)
    references.append(reference)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/50 done")

# compute ROUGE-L
rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=predictions, references=references)
print(f"\n--- EVALUATION RESULTS ---")
print(f"ROUGE-L : {scores['rougeL']:.4f}")
print(f"ROUGE-1 : {scores['rouge1']:.4f}")
print(f"ROUGE-2 : {scores['rouge2']:.4f}")

# compute BERTScore
bertscore = evaluate.load("bertscore")
bert_scores = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)
avg_f1 = sum(bert_scores["f1"]) / len(bert_scores["f1"])
print(f"BERTScore F1: {avg_f1:.4f}")

# save results
results = {
    "rouge1":       round(scores["rouge1"], 4),
    "rouge2":       round(scores["rouge2"], 4),
    "rougeL":       round(scores["rougeL"], 4),
    "bertscore_f1": round(avg_f1, 4),
    "sample_output": {
        "transcript": test_data[0]["transcript"][:300],
        "generated":  predictions[0],
        "reference":  references[0]
    }
}
save_path = os.path.join(DRIVE_BASE, "results/eval_results.json")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {save_path}")

Loading fine-tuned model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded
Running evaluation on 50 test samples...
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done



--- EVALUATION RESULTS ---
ROUGE-L : 0.2100
ROUGE-1 : 0.2494
ROUGE-2 : 0.0895


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore F1: 0.8476

Results saved to /content/drive/MyDrive/EarningScribe/results/eval_results.json


In [1]:
!pip install transformers peft accelerate datasets \
             evaluate rouge-score bert-score -U -q
!pip install bitsandbytes -q
print("Done")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
Done


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists("/content/EarningScribe"):
    !git clone https://github.com/jiviteshhp/EarningScribe.git /content/EarningScribe

%cd /content/EarningScribe
print("Ready")

Mounted at /content/drive
Cloning into '/content/EarningScribe'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 0), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 5.89 KiB | 5.89 MiB/s, done.
/content/EarningScribe
Ready


In [9]:
import os
DRIVE_BASE = "/content/drive/MyDrive/EarningScribe"

for root, dirs, files in os.walk(DRIVE_BASE):
    for f in files:
        print(os.path.join(root, f))

In [10]:
import json, os
from datasets import load_dataset

DATA_BASE = "/content/EarningScribe/data/processed"
os.makedirs(DATA_BASE, exist_ok=True)

print("Downloading dataset... ")
dataset = load_dataset("lamini/earnings-calls-qa")

def clean_text(text):
    lines = text.split("\n")
    skip_phrases = ["operator","safe harbor","forward-looking","this concludes",
                    "thank you for joining","ladies and gentlemen","this call is being recorded"]
    cleaned = []
    for line in lines:
        line = line.strip()
        if len(line) < 15: continue
        if any(line.lower().startswith(p) for p in skip_phrases): continue
        cleaned.append(line)
    return " ".join(cleaned)

all_samples, seen = [], set()
for i, item in enumerate(dataset["train"]):
    answer = item["answer"].strip()
    transcript = clean_text(item["transcript"])
    if "i do not know" in answer.lower(): continue
    if len(transcript.split()) < 50: continue
    if len(answer.split()) < 5: continue
    fp = transcript[:200]
    if fp in seen: continue
    seen.add(fp)
    all_samples.append({"id": f"sample_{i}", "transcript": transcript,
                        "ticker": item.get("ticker",""), "date": item.get("date",""),
                        "output": {"financial_insight": item["question"].strip(), "detail": answer}})
    if len(all_samples) == 5000: break

total = len(all_samples)
splits = {"train": all_samples[:int(total*0.8)],
          "validation": all_samples[int(total*0.8):int(total*0.9)],
          "test": all_samples[int(total*0.9):]}

for name, data in splits.items():
    with open(os.path.join(DATA_BASE, f"{name}.json"), "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(data)} samples -> {name}.json")

print("Done ")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

filtered_predictions.jsonl:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/860164 [00:00<?, ? examples/s]

Saved 4000 samples -> train.json
Saved 500 samples -> validation.json
Saved 500 samples -> test.json
Done ✓


In [12]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
DATA_BASE  = "/content/EarningScribe/data/processed"

with open(os.path.join(DATA_BASE, "test.json")) as f:
    test_data = json.load(f)
print(f"Test samples loaded: {len(test_data)} ✓")

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.float16
)
base_model.eval()
print("Base model loaded ")

Test samples loaded: 500 ✓
Loading base model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded ✓


In [13]:
def make_prompt(transcript):
    return f"""You are a senior financial analyst.

TRANSCRIPT:
{transcript[:600]}

Generate a JSON report with these fields:
- company_summary: 2-3 sentence plain English summary
- key_metrics: list of strings like "Revenue: $4.2B (+12% YoY)"
- guidance: one plain English sentence about next quarter
- risks: list of risk strings
- sentiment: one word, positive or neutral or negative

REPORT:"""


def generate(model, tokenizer, transcript):
    prompt  = make_prompt(transcript)
    inputs  = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=768).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


print("Generating predictions for 50 samples...")
base_predictions = []
base_references  = []

for i, item in enumerate(test_data[:50]):
    base_predictions.append(generate(base_model, tokenizer, item["transcript"]))
    base_references.append(item["output"]["detail"])
    if (i+1) % 10 == 0:
        print(f"  {i+1}/50 done")

print("Generation complete")

Generating predictions for 50 samples...
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
Generation complete


In [14]:
from evaluate import load as eval_load

print("Computing ROUGE scores...")
rouge  = eval_load("rouge")
scores = rouge.compute(predictions=base_predictions, references=base_references)

print("Computing BERTScore...")
bertscore  = eval_load("bertscore")
bert       = bertscore.compute(
    predictions=base_predictions,
    references=base_references,
    lang="en",
    model_type="distilbert-base-uncased",
    rescale_with_baseline=False
)
avg_f1 = sum(bert["f1"]) / len(bert["f1"])

print("\n--- BASE MODEL RESULTS ---")
print(f"ROUGE-1    : {scores['rouge1']:.4f}")
print(f"ROUGE-2    : {scores['rouge2']:.4f}")
print(f"ROUGE-L    : {scores['rougeL']:.4f}")
print(f"BERTScore  : {avg_f1:.4f}")

Computing ROUGE scores...


Computing BERTScore...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- BASE MODEL RESULTS ---
ROUGE-1    : 0.1333
ROUGE-2    : 0.0244
ROUGE-L    : 0.0922
BERTScore  : 0.6798


In [15]:


DRIVE_BASE = "/content/drive/MyDrive/EarningScribe"
os.makedirs(os.path.join(DRIVE_BASE, "results"), exist_ok=True)

base_results = {
    "rouge1":       0.1333,
    "rouge2":       0.0244,
    "rougeL":       0.0922,
    "bertscore_f1": 0.6798
}

ft_results = {
    "rouge1":       0.2494,
    "rouge2":       0.0895,
    "rougeL":       0.2100,
    "bertscore_f1": 0.8476
}

print("=" * 57)
print(f"{'Metric':<20} {'Base':>10} {'Fine-tuned':>12} {'Improvement':>12}")
print("=" * 57)
for metric in ["rouge1", "rouge2", "rougeL", "bertscore_f1"]:
    base = base_results[metric]
    ft   = ft_results[metric]
    imp  = (ft - base) / base * 100
    print(f"{metric:<20} {base:>10.4f} {ft:>12.4f} {imp:>+11.1f}%")
print("=" * 57)

with open(os.path.join(DRIVE_BASE, "results/eval_comparison.json"), "w") as f:
    json.dump({"base": base_results, "finetuned": ft_results}, f, indent=2)
print("\nSaved to Drive")

Metric                     Base   Fine-tuned  Improvement
rouge1                   0.1333       0.2494       +87.1%
rouge2                   0.0244       0.0895      +266.8%
rougeL                   0.0922       0.2100      +127.8%
bertscore_f1             0.6798       0.8476       +24.7%

Saved to Drive
